# Notebook 10 — Concept Tagging via Wikipedia + spaCy NER

For each work in `works_corpus.csv`, extract cultural literacy concepts using:
1. **Wikipedia article text** (fetched via MediaWiki API, cached to disk)
2. **spaCy `en_core_web_lg` NER** — PERSON, GPE, LOC, EVENT, ORG, WORK_OF_ART
3. **Wikipedia category parsing** — e.g. `Films about World War II` → `World War II`
4. **Work metadata** — director/author tagged at relevance 3

Results matched back to `cultural_salience_composite_v2.csv`.

**Output:** `data/work_concept_tags.csv`, `data/work_concept_yield.csv`

In [1]:
import pandas as pd
import numpy as np
import re
import json
import time
import unicodedata
import requests
from pathlib import Path
from collections import Counter
import spacy
import warnings
warnings.filterwarnings('ignore')

DATA_DIR   = Path('../data')
CACHE_DIR  = DATA_DIR / 'cache' / 'nb10'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

WIKI_API   = 'https://en.wikipedia.org/w/api.php'
WD_API     = 'https://www.wikidata.org/w/api.php'
HEADERS    = {'User-Agent': 'CulturalLiteracyProject/1.0 (research)'}
SLEEP_SEC  = 0.15   # polite delay between API requests

# NER entity types to keep
KEEP_TYPES = {'PERSON', 'GPE', 'LOC', 'ORG', 'EVENT', 'WORK_OF_ART', 'NORP', 'FAC'}

# Map spaCy label → concept_type string
LABEL_TYPE = {
    'PERSON': 'person', 'GPE': 'place', 'LOC': 'place', 'FAC': 'place',
    'ORG': 'other', 'EVENT': 'event', 'WORK_OF_ART': 'work',
    'NORP': 'movement',
}


def normalize(text):
    if not isinstance(text, str):
        return ''
    s = unicodedata.normalize('NFKD', text)
    s = ''.join(c for c in s if not unicodedata.combining(c))
    s = s.lower().strip()
    s = re.sub(r'^(the|a|an)\s+', '', s)
    s = re.sub(r"[,;:!?'\"()\[\]]", '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


print('Loading spaCy en_core_web_lg...')
nlp = spacy.load('en_core_web_lg', disable=['parser', 'lemmatizer'])
print(f'spaCy loaded: {nlp.meta["name"]} v{nlp.meta["version"]}')

Loading spaCy en_core_web_lg...


spaCy loaded: core_web_lg v3.8.0


## 1. Load Data

In [2]:
works = pd.read_csv(DATA_DIR / 'works_corpus.csv')
print(f'Works corpus: {len(works):,}  (films: {(works["medium"]=="film").sum()}, novels: {(works["medium"]=="novel").sum()})')
print(f'Works with QID: {works["qid"].notna().sum():,}')

# Composite dataset for matching
composite = pd.read_csv(
    DATA_DIR / 'cultural_salience_composite_v2.csv',
    dtype={'wiki_title': str, 'qid': str},
    low_memory=False,
)
composite['norm'] = composite['concept'].apply(normalize)
comp_top = composite[composite['composite_score'] >= 40].drop_duplicates('norm').set_index('norm')

comp_display = composite[composite['composite_score'] >= 40].copy()
comp_display['norm_disp'] = comp_display['display_name'].apply(normalize)
comp_display = comp_display.drop_duplicates('norm_disp').set_index('norm_disp')

print(f'Composite concepts (score>=40): {len(comp_top):,}')

Works corpus: 883  (films: 444, novels: 439)
Works with QID: 347


Composite concepts (score>=40): 141,448


## 2. Wikipedia Fetch Helpers

In [3]:
def _wiki_get(params):
    """GET the Wikipedia API with retries."""
    params.setdefault('format', 'json')
    for attempt in range(3):
        try:
            r = requests.get(WIKI_API, params=params, headers=HEADERS, timeout=15)
            r.raise_for_status()
            return r.json()
        except Exception:
            if attempt < 2:
                time.sleep(2 ** attempt)
    return {}


def _wd_get(params):
    """GET the Wikidata API with retries."""
    params.setdefault('format', 'json')
    for attempt in range(3):
        try:
            r = requests.get(WD_API, params=params, headers=HEADERS, timeout=15)
            r.raise_for_status()
            return r.json()
        except Exception:
            if attempt < 2:
                time.sleep(2 ** attempt)
    return {}


def qid_to_enwiki_title(qid):
    """Return the English Wikipedia article title for a Wikidata QID."""
    if not qid or not str(qid).startswith('Q'):
        return None
    data = _wd_get({'action': 'wbgetentities', 'ids': qid,
                    'props': 'sitelinks', 'sitefilter': 'enwiki'})
    try:
        return data['entities'][qid]['sitelinks']['enwiki']['title']
    except (KeyError, TypeError):
        return None


def search_enwiki(query, limit=1):
    """Search Wikipedia, return best-matching title."""
    data = _wiki_get({'action': 'query', 'list': 'search',
                      'srsearch': query, 'srlimit': limit})
    try:
        results = data['query']['search']
        return results[0]['title'] if results else None
    except (KeyError, IndexError):
        return None


def fetch_article(wiki_title, max_chars=4000):
    """
    Fetch plain-text extract + categories for a Wikipedia title.
    Returns dict: {extract, categories, found}.
    """
    data = _wiki_get({
        'action': 'query',
        'titles': wiki_title,
        'prop': 'extracts|categories',
        'exintro': True,          # intro section only
        'explaintext': True,      # plain text, no HTML
        'cllimit': 50,
        'clshow': '!hidden',      # skip hidden maintenance categories
    })
    try:
        pages = data['query']['pages']
        page  = next(iter(pages.values()))
        if 'missing' in page:
            return {'found': False}
        extract = page.get('extract', '')[:max_chars]
        cats = [c['title'].replace('Category:', '') for c in page.get('categories', [])]
        return {'found': True, 'wiki_title': wiki_title, 'extract': extract, 'categories': cats}
    except (KeyError, StopIteration):
        return {'found': False}


def get_wiki_data(row):
    """
    For a work row: look up Wikipedia, fetch article, cache result.
    Returns cached dict.
    """
    work_id    = row['work_id']
    cache_path = CACHE_DIR / f'{work_id}.json'

    if cache_path.exists():
        with open(cache_path) as f:
            return json.load(f)

    medium  = row['medium']
    title   = str(row['title'])
    creator = str(row.get('director' if medium == 'film' else 'author', '')).strip()
    if creator == 'nan':
        creator = ''
    qid = row.get('qid')

    # 1. Try QID → exact Wikipedia title
    wiki_title = None
    if pd.notna(qid) and str(qid).startswith('Q'):
        wiki_title = qid_to_enwiki_title(str(qid))
        time.sleep(SLEEP_SEC)

    # 2. Fallback: search Wikipedia
    if not wiki_title:
        query = f'{title} {medium}'
        if creator:
            query += f' {creator}'
        wiki_title = search_enwiki(query)
        time.sleep(SLEEP_SEC)

    # 3. Fetch article
    result = {'work_id': work_id, 'title': title}
    if wiki_title:
        article = fetch_article(wiki_title)
        result.update(article)
        time.sleep(SLEEP_SEC)
    else:
        result['found'] = False

    with open(cache_path, 'w') as f:
        json.dump(result, f)
    return result


print('Wikipedia helpers ready.')

Wikipedia helpers ready.


## 3. Concept Extraction

In [4]:
# Patterns to strip from category strings to get a concept
_CAT_STRIP = re.compile(
    r'^(Films?|Novels?|Books?|Stories?|Works?|American|British|French|German|Italian|'
    r'English|Japanese|Soviet|Spanish|Chinese)\s+(about|based on|directed by|written by|'
    r'set in|featuring|depicting|starring|adaptat\w+|by)\s+', re.I
)
_CAT_PREFIX = re.compile(
    r'^(Films?|Novels?|Books?|Stories?|Works?)\s+(about|set in|from|of|during|in)\s+', re.I
)
_CAT_JUNK   = re.compile(
    r'(stub|\d{4}s?\s+in\s+|fictional|Wikipedia|articles|categories|pages|redirects)', re.I
)


def parse_categories(cats):
    """
    Convert Wikipedia category strings into (concept, type) pairs.
    e.g. 'Films about World War II' -> ('World War II', 'event')
         'Films set in Ancient Rome' -> ('Ancient Rome', 'place')
    """
    results = []
    for cat in cats:
        if _CAT_JUNK.search(cat):
            continue
        # Strip leading film/novel qualifiers
        concept = _CAT_STRIP.sub('', cat).strip()
        concept = _CAT_PREFIX.sub('', concept).strip()
        # Remove trailing qualifiers like '(film)' '(novel)'
        concept = re.sub(r'\s*\([^)]+\)\s*$', '', concept).strip()
        if len(concept) < 3 or len(concept) > 80:
            continue
        # Guess type
        ctype = 'theme'
        if re.search(r'\bwar\b|\bworld war\b|\bholo\w+|\brevolution|\bbattle\b', concept, re.I):
            ctype = 'event'
        elif re.search(r'\b(century|era|period|age|empire|kingdom|republic)\b', concept, re.I):
            ctype = 'event'
        elif concept[0].isupper() and ' ' not in concept:
            ctype = 'place'   # single capitalized word — likely a place
        results.append((concept, ctype))
    return results


def extract_concepts(wiki_data, row):
    """
    Return list of dicts: {display_concept, concept_norm, concept_type, relevance}
    """
    concepts = []
    seen_norms = set()

    def add(display, ctype, relevance):
        norm = normalize(display)
        if not norm or norm in seen_norms:
            return
        seen_norms.add(norm)
        concepts.append({
            'display_concept': display,
            'concept_norm'   : norm,
            'concept_type'   : ctype,
            'relevance'      : relevance,
        })

    # ── 1. Work creator at relevance 3 ──────────────────────────────────────
    medium  = row['medium']
    creator = str(row.get('director' if medium == 'film' else 'author', '')).strip()
    if creator and creator != 'nan':
        add(creator, 'person', 3)

    # ── 2. spaCy NER on article extract ─────────────────────────────────────
    extract = wiki_data.get('extract', '')
    if extract:
        doc = nlp(extract)
        ent_count  = Counter()
        ent_labels = {}
        for ent in doc.ents:
            if ent.label_ not in KEEP_TYPES:
                continue
            text = ent.text.strip()
            if len(text) < 2 or len(text) > 60:
                continue
            # Skip if entity text matches the work title closely
            if normalize(text) == normalize(row['title']):
                continue
            ent_count[text] += 1
            ent_labels[text] = ent.label_

        for ent_text, count in ent_count.most_common(30):
            relevance = 3 if count >= 5 else (2 if count >= 2 else 1)
            ctype = LABEL_TYPE.get(ent_labels[ent_text], 'other')
            add(ent_text, ctype, relevance)

    # ── 3. Wikipedia category concepts ──────────────────────────────────────
    for concept, ctype in parse_categories(wiki_data.get('categories', [])):
        add(concept, ctype, 2)   # category-derived → relevance 2

    return concepts


print('Concept extraction helpers ready.')

Concept extraction helpers ready.


## 4. Process All Works

In [5]:
already = sum(1 for wid in works['work_id'] if (CACHE_DIR / f'{wid}.json').exists())
print(f'Already cached : {already:,} / {len(works):,}')
print(f'To fetch       : {len(works) - already:,}  (≈{(len(works)-already)*SLEEP_SEC*3:.0f}s at {SLEEP_SEC}s/req × 3 reqs/work)')

wiki_results = {}
t0 = time.time()

for i, (_, row) in enumerate(works.iterrows()):
    wiki_results[row['work_id']] = get_wiki_data(row)

    if (i + 1) % 100 == 0:
        elapsed = time.time() - t0
        rate    = (i + 1) / elapsed
        eta     = (len(works) - i - 1) / rate
        found   = sum(1 for v in wiki_results.values() if v.get('found'))
        print(f'  [{i+1:4d}/{len(works)}] {elapsed:4.0f}s  '
              f'rate={rate:.1f}/s  ETA~{eta:.0f}s  found={found}')

elapsed = time.time() - t0
found   = sum(1 for v in wiki_results.values() if v.get('found'))
print(f'\nDone. {elapsed:.0f}s  Wikipedia found: {found}/{len(works)}')

Already cached : 0 / 883
To fetch       : 883  (≈397s at 0.15s/req × 3 reqs/work)


  [ 100/883]  124s  rate=0.8/s  ETA~972s  found=98


  [ 200/883]  269s  rate=0.7/s  ETA~920s  found=198


  [ 300/883]  436s  rate=0.7/s  ETA~848s  found=296


  [ 400/883]  581s  rate=0.7/s  ETA~702s  found=394


  [ 500/883]  731s  rate=0.7/s  ETA~560s  found=491


  [ 600/883]  882s  rate=0.7/s  ETA~416s  found=577


  [ 700/883] 1026s  rate=0.7/s  ETA~268s  found=673


  [ 800/883] 1183s  rate=0.7/s  ETA~123s  found=764



Done. 1308s  Wikipedia found: 845/883


## 5. Run NER + Category Extraction

In [6]:
rows = []
work_meta_map = works.set_index('work_id').to_dict('index')

for work_id, wiki_data in wiki_results.items():
    row = work_meta_map[work_id]
    row['work_id'] = work_id
    for c in extract_concepts(wiki_data, row):
        c['work_id'] = work_id
        rows.append(c)

tags = pd.DataFrame(rows)
print(f'Raw concept tags: {len(tags):,}')
print(f'Unique works    : {tags["work_id"].nunique()}')
print(f'Unique concepts : {tags["concept_norm"].nunique():,}')
print()
print('Concept type distribution:')
print(tags['concept_type'].value_counts().head(10).to_string())
print('\nRelevance distribution:')
print(tags['relevance'].value_counts().sort_index().to_string())
print('\nTags per work:')
per_work = tags.groupby('work_id').size()
print(per_work.describe(percentiles=[.25, .5, .75, .95]).to_string())

Raw concept tags: 33,166
Unique works    : 881
Unique concepts : 16,482

Concept type distribution:
concept_type
theme       17289
person       6842
other        2902
place        2501
event        1459
movement     1239
work          934

Relevance distribution:
relevance
1    11724
2    20486
3      956

Tags per work:
count    881.000000
mean      37.645857
std       22.165532
min        1.000000
25%       19.000000
50%       33.000000
75%       56.000000
95%       77.000000
max       81.000000


## 6. Match Tags to Composite Dataset

In [7]:
def match_concept(norm):
    """Return (matched_concept, matched_qid, matched_score) or (nan, nan, nan)."""
    if norm in comp_top.index:
        r = comp_top.loc[norm]
        if isinstance(r, pd.DataFrame): r = r.iloc[0]
        return r.get('concept', norm), r.get('qid', np.nan), float(r.get('composite_score', np.nan))
    if norm in comp_display.index:
        r = comp_display.loc[norm]
        if isinstance(r, pd.DataFrame): r = r.iloc[0]
        return r.get('concept', norm), r.get('qid', np.nan), float(r.get('composite_score', np.nan))
    return np.nan, np.nan, np.nan


unique_norms = tags['concept_norm'].unique()
print(f'Matching {len(unique_norms):,} unique concepts...')
match_cache = {n: match_concept(n) for n in unique_norms}

tags['matched_concept'] = tags['concept_norm'].map(lambda n: match_cache[n][0])
tags['matched_qid']     = tags['concept_norm'].map(lambda n: match_cache[n][1])
tags['matched_score']   = tags['concept_norm'].map(lambda n: match_cache[n][2]).astype(float)

n_matched = tags['matched_concept'].notna().sum()
n_unique_matched = sum(1 for v in match_cache.values() if pd.notna(v[0]))
print(f'Matched {n_matched:,} / {len(tags):,} tags ({100*n_matched/len(tags):.1f}%)')
print(f'Unique concepts matched: {n_unique_matched:,} / {len(unique_norms):,} '
      f'({100*n_unique_matched/len(unique_norms):.1f}%)')

Matching 16,482 unique concepts...


Matched 9,263 / 33,166 tags (27.9%)
Unique concepts matched: 3,460 / 16,482 (21.0%)


## 7. Export

In [8]:
work_meta = works[['work_id', 'title', 'medium', 'year']]
export = tags.merge(work_meta, on='work_id', how='left')[[
    'work_id', 'title', 'medium', 'year',
    'display_concept', 'concept_norm', 'concept_type', 'relevance',
    'matched_concept', 'matched_qid', 'matched_score',
]]

out_path = DATA_DIR / 'work_concept_tags.csv'
export.to_csv(out_path, index=False)
print(f'Saved {len(export):,} rows → {out_path}')
print()
print(export.head(20).to_string())

Saved 33,166 rows → ..\data\work_concept_tags.csv

   work_id  title medium    year                       display_concept                          concept_norm concept_type  relevance matched_concept matched_qid  matched_score
0   W00311  Ikiru   film  1952.0                        Akira Kurosawa                        akira kurosawa       person          3  akira kurosawa       Q8006      74.263883
1   W00311  Ikiru   film  1952.0                               To Live                               to live         work          1         to live         NaN      51.754301
2   W00311  Ikiru   film  1952.0                              Japanese                              japanese     movement          1        japanese         NaN      88.032795
3   W00311  Ikiru   film  1952.0                     Shinobu Hashimoto                     shinobu hashimoto       person          1             NaN         NaN            NaN
4   W00311  Ikiru   film  1952.0                           Hideo Ogun

## 8. Summary & Concept Yield

In [9]:
print('=' * 65)
print('CONCEPT TAGGING SUMMARY')
print('=' * 65)
print(f'  Works tagged           : {export["work_id"].nunique():,} / {len(works):,}')
print(f'  Total tag rows         : {len(export):,}')
print(f'  Unique LLM concepts    : {export["concept_norm"].nunique():,}')
print(f'  Match rate             : {100*export["matched_concept"].notna().mean():.1f}%')

tpw = export.groupby('work_id').size()
print(f'\n  Tags per work  mean={tpw.mean():.1f}  median={tpw.median():.0f}  '
      f'min={tpw.min()}  max={tpw.max()}')

print(f'\nTop 30 concepts by number of works:')
top_c = (
    export[export['matched_concept'].notna()]
    .groupby('matched_concept')
    .agg(n_works=('work_id','nunique'), avg_rel=('relevance','mean'),
         score=('matched_score','first'))
    .sort_values('n_works', ascending=False).head(30)
)
print(top_c.to_string(float_format='{:.1f}'.format))

# ── Concept yield ────────────────────────────────────────────────────────────
matched = export[export['matched_concept'].notna()].copy()
matched['weighted'] = matched['relevance'] * matched['matched_score']
yield_df = (
    matched.groupby('work_id')
    .agg(n_concepts=('concept_norm','nunique'),
         n_matched=('matched_concept','nunique'),
         concept_yield=('weighted','sum'),
         avg_score=('matched_score','mean'))
    .reset_index()
    .merge(works[['work_id','title','medium','year','importance_score']], on='work_id', how='left')
    .sort_values('concept_yield', ascending=False)
)

print('\nTop 20 films by concept yield:')
print(yield_df[yield_df['medium']=='film'].head(20)[
    ['title','year','n_matched','concept_yield','importance_score']
].to_string(index=False, float_format='{:.1f}'.format))

print('\nTop 20 novels by concept yield:')
print(yield_df[yield_df['medium']=='novel'].head(20)[
    ['title','year','n_matched','concept_yield','importance_score']
].to_string(index=False, float_format='{:.1f}'.format))

yield_df.to_csv(DATA_DIR / 'work_concept_yield.csv', index=False)
print(f'\nSaved yield summary → {DATA_DIR}/work_concept_yield.csv')

CONCEPT TAGGING SUMMARY
  Works tagged           : 881 / 883
  Total tag rows         : 33,166
  Unique LLM concepts    : 16,482
  Match rate             : 27.9%

  Tags per work  mean=37.6  median=33  min=1  max=81

Top 30 concepts by number of works:
                      n_works  avg_rel  score
matched_concept                              
american                  429      1.2   91.3
united states             215      1.2   87.2
new york city              86      1.6   83.2
academy awards             83      1.0   40.6
british                    79      1.3   81.1
french                     68      1.5   89.7
library of congress        67      1.0   99.4
time                       59      1.1   98.8
london                     59      1.6   95.7
academy award              47      1.1   66.1
england                    45      1.4   93.6
paris                      45      1.5   97.5
france                     44      1.4   95.1
united kingdom             44      1.1   76.8
los angeles


Saved yield summary → ..\data/work_concept_yield.csv
